In [0]:
# Databricks notebook source

# COMMAND ----------
# 1. IMPORTS

from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import uuid


# COMMAND ----------
# 2. CONFIGURATION

PREDICTIONS_TABLE = (
    "fraud_mlops.gold.production_predictions"
)

GROUND_TRUTH_TABLE = (
    "fraud_mlops.gold.production_ground_truth"
)

OUTPUT_TABLE = (
    "fraud_mlops.gold.model_performance_metrics"
)


# COMMAND ----------
# 3. JOIN PREDICTIONS WITH AVAILABLE LABELS

performance_df = (
    spark.table(PREDICTIONS_TABLE)
    .alias("p")

    .join(
        spark.table(GROUND_TRUTH_TABLE)
        .filter(
            F.col("label_status") == "AVAILABLE"
        )
        .alias("g"),

        F.col("p.transaction_id")
        == F.col("g.transaction_id"),

        "inner"
    )

    .select(
        F.col("p.transaction_id"),
        F.col("p.fraud_probability"),
        F.col("p.fraud_prediction"),
        F.col("p.model_version"),
        F.col("p.deployment_name"),
        F.col("g.is_fraud")
    )
)


labeled_rows = performance_df.count()

print(
    "Labeled prediction rows:",
    labeled_rows
)


if labeled_rows == 0:
    raise ValueError(
        "No labeled production predictions available."
    )


# COMMAND ----------
# 4. CONFUSION MATRIX

counts = (
    performance_df
    .agg(

        F.sum(
            F.when(
                (F.col("fraud_prediction") == 1)
                &
                (F.col("is_fraud") == 1),
                1
            ).otherwise(0)
        ).alias("tp"),

        F.sum(
            F.when(
                (F.col("fraud_prediction") == 1)
                &
                (F.col("is_fraud") == 0),
                1
            ).otherwise(0)
        ).alias("fp"),

        F.sum(
            F.when(
                (F.col("fraud_prediction") == 0)
                &
                (F.col("is_fraud") == 0),
                1
            ).otherwise(0)
        ).alias("tn"),

        F.sum(
            F.when(
                (F.col("fraud_prediction") == 0)
                &
                (F.col("is_fraud") == 1),
                1
            ).otherwise(0)
        ).alias("fn")
    )
    .first()
)


tp = counts["tp"]
fp = counts["fp"]
tn = counts["tn"]
fn = counts["fn"]


print("TP:", tp)
print("FP:", fp)
print("TN:", tn)
print("FN:", fn)


# COMMAND ----------
# 5. PRECISION, RECALL, F1, FPR

precision = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else 0.0
)

recall = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0.0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

fpr = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)


# COMMAND ----------
# 6. PR-AUC

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="fraud_probability",
    metricName="areaUnderPR"
)

pr_auc = pr_evaluator.evaluate(
    performance_df
)


# COMMAND ----------
# 7. ROC-AUC

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="fraud_probability",
    metricName="areaUnderROC"
)

roc_auc = roc_evaluator.evaluate(
    performance_df
)


# COMMAND ----------
# 8. DISPLAY METRICS

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("PR-AUC:", pr_auc)
print("ROC-AUC:", roc_auc)
print("FPR:", fpr)


# COMMAND ----------
# 9. CREATE MONITORING RESULT

monitoring_run_id = str(
    uuid.uuid4()
)


model_version = (
    performance_df
    .select("model_version")
    .first()["model_version"]
)


deployment_name = (
    performance_df
    .select("deployment_name")
    .first()["deployment_name"]
)


fraud_rows = tp + fn


metrics_df = spark.createDataFrame(
    [
        (
            monitoring_run_id,
            model_version,
            deployment_name,

            labeled_rows,
            fraud_rows,

            tp,
            fp,
            tn,
            fn,

            float(precision),
            float(recall),
            float(f1),
            float(pr_auc),
            float(roc_auc),
            float(fpr)
        )
    ],

    [
        "monitoring_run_id",
        "model_version",
        "deployment_name",

        "labeled_rows",
        "fraud_rows",

        "true_positive",
        "false_positive",
        "true_negative",
        "false_negative",

        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc",
        "fpr"
    ]
)


metrics_df = (
    metrics_df
    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)


# COMMAND ----------
# 10. DISPLAY RESULT

display(metrics_df)


# COMMAND ----------
# 11. SAVE PERFORMANCE HISTORY

(
    metrics_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(OUTPUT_TABLE)
)


# COMMAND ----------
# 12. COMPLETION MESSAGE

print(
    f"Performance monitoring completed.\n"
    f"Rows evaluated: {labeled_rows}\n"
    f"Precision: {precision:.6f}\n"
    f"Recall: {recall:.6f}\n"
    f"F1: {f1:.6f}\n"
    f"PR-AUC: {pr_auc:.6f}\n"
    f"ROC-AUC: {roc_auc:.6f}\n"
    f"FPR: {fpr:.6f}\n"
    f"Output table: {OUTPUT_TABLE}"
)